# Día 6 · Unidad 6 — Código profesional (I): módulos, paquetes y PEP 8

**Objetivos de aprendizaje**
- Explicar qué son los módulos y los paquetes de Python y por qué la programación modular facilita el mantenimiento de un proyecto.
- Usar las distintas variantes de `import` (`import x`, `from x import y`, alias) y entender qué hace cada una con el *namespace*.
- Entender cómo busca Python un módulo (`sys.path`) y qué significa `__name__ == "__main__"`.
- Organizar código en paquetes (`__init__.py`, `__all__`) en vez de un único notebook o script gigante.
- Aplicar las convenciones de estilo PEP 8 (nombrado, espaciado, longitud de línea) a tu propio código.

**Duración estimada:** 90-105 minutos.

**Requisitos previos:** Día 5 (funciones y clases). A partir de este notebook usarás archivos `.py` además del propio notebook — todos están ya en esta misma carpeta (`dia_06_codigo_profesional/`).

## 1. Motivación

Hasta ahora, todo el código de un día de curso vive dentro de notebooks. Funciona para practicar, pero un proyecto real de la aseguradora — un motor de tarificación, un pipeline de reporting regulatorio — no cabe en un único notebook: se vuelve imposible de mantener, revisar o reutilizar.

La **programación modular** consiste en dividir un programa grande en piezas más pequeñas y manejables: los **módulos** (un archivo `.py`) y los **paquetes** (una carpeta de módulos relacionados). Ventajas:

- **Simplicidad**: cada módulo se ocupa de una porción pequeña del problema (p. ej. `tarificacion.py` solo calcula primas, `validaciones.py` solo valida datos de entrada).
- **Mantenimiento**: si los módulos están bien separados, modificar uno no debería romper los demás.
- **Reutilización**: una función escrita una vez en un módulo se importa desde cualquier otro script o notebook, sin copiar y pegar código.
- **Organización de nombres (*namespacing*)**: cada módulo tiene su propio espacio de nombres, así que puedes tener una función `calcular()` en `tarificacion.py` y otra `calcular()` en `provisiones.py` sin que choquen entre sí.

> 📊 En Excel esto sería como separar un libro gigante de 40 pestañas en varios libros más pequeños con un propósito claro cada uno (`Tarificacion.xlsx`, `Siniestros.xlsx`), en vez de una única hoja `Hoja34` donde nadie recuerda ya qué hace cada columna.

## 2. Módulos de Python

Un **módulo** es, en el caso más habitual, simplemente un archivo `.py` que contiene código Python: variables, funciones, clases. No hace falta nada especial para crear uno — basta con guardar un archivo con extensión `.py`.

En esta misma carpeta tienes un archivo `modulo_test.py` con este contenido:

```python
s = "String dentro de un módulo"

l = [100, 200, 300]

def func_modulo(name: str):
    print(f"Nombre dentro de una función de módulo {name}")

class Poliza:
    pass
```

Si intentas acceder a `s` directamente desde el notebook, sin haberlo importado, obtienes un `NameError`: el notebook y el archivo `modulo_test.py` tienen *namespaces* completamente separados.

In [ ]:
try:
    print(s)
except NameError as e:
    print(f"{type(e).__name__}: {e}")

Para acceder a su contenido, hace falta **importar** el módulo:

In [ ]:
import modulo_test

print(modulo_test.s)
print(modulo_test.l)
modulo_test.func_modulo(name="Actuaria")

## 3. La ruta de búsqueda de módulos: `sys.path`

¿Cómo sabe Python dónde buscar `modulo_test.py` cuando ejecutas `import modulo_test`? Recorre una lista de directorios almacenada en `sys.path`, construida a partir de:

1. El directorio desde el que se ejecuta el notebook o script (o el directorio actual, si el intérprete es interactivo).
2. Los directorios de la variable de entorno `PYTHONPATH`, si existe.
3. Los directorios propios de la instalación de Python (donde viven `pandas`, `numpy`, etc.).

In [ ]:
import sys

for ruta in sys.path[:6]:
    print(ruta)

Por eso `import modulo_test` funciona en este notebook: `modulo_test.py` está en la misma carpeta que el notebook (opción 1). Si movieras el notebook a otra carpeta sin mover también `modulo_test.py`, el `import` fallaría con `ModuleNotFoundError`.

> ⚠️ **Error típico**: copiar un notebook a otra carpeta y olvidar copiar también los `.py` de los que depende. El notebook funcionaba "en tu máquina" y deja de funcionar en la de un compañero — casi siempre es un problema de `sys.path`, no de que el código esté mal escrito.

Es posible ampliar `sys.path` en tiempo de ejecución con `sys.path.append(...)`, pero es un parche: la solución robusta es organizar el proyecto en paquetes (sección 6) o instalarlo como tal, no ir añadiendo rutas sueltas.

## 4. La cláusula `import` en sus variantes

### `import <módulo>`

Coloca el **nombre del módulo** en el *namespace* global; el contenido queda accesible con el prefijo `<módulo>.`

In [ ]:
import modulo_test

modulo_test.s

### `from <módulo> import <nombre>`

Importa objetos concretos directamente al *namespace* global — sin necesidad del prefijo.

In [ ]:
from modulo_test import s

s

> ⚠️ **Error típico**: con esta forma puedes **sobrescribir** una variable que ya tenías definida, sin ningún aviso. Si ya tenías una variable `l` con otro significado, `from modulo_test import l` la pisa silenciosamente.

In [ ]:
l = "Esto no es una lista, es un texto que ya tenía yo"
print(l)

from modulo_test import l
print(l)  # se ha sobrescrito sin avisar

> **Nota**: existe también `from <módulo> import *`, que importa *todo* lo público del módulo. Se considera mala práctica casi siempre — no sabes qué nombres estás trayendo al *namespace* ni de dónde vienen, y el riesgo de colisión aumenta. Evítalo salvo casos muy concretos (p. ej. algunos paquetes de configuración).

### Alias: `import <módulo> as <alias>` y `from <módulo> import <nombre> as <alias>`

Útil para nombres largos o para evitar colisiones — es el patrón que ya usas con `import pandas as pd` o `import numpy as np`.

In [ ]:
import modulo_test as mt
from modulo_test import s as string_del_modulo

print(mt.l)
print(string_del_modulo)

### La función `dir()`

Devuelve los nombres definidos en un *namespace*. Sin argumentos, los del *namespace* actual; con un módulo como argumento, los del módulo.

In [ ]:
[nombre for nombre in dir(modulo_test) if not nombre.startswith("__")]

**Para ti:** importa el módulo estándar `math` con alias `m` y usa `dir(m)` para encontrar el nombre de la función que calcula la raíz cuadrada. Compruébalo calculando `m.sqrt(144)`.

In [ ]:
# Tu código aquí


## 5. Ejecutar módulos como script: `__name__ == "__main__"`

Cualquier archivo `.py` es, a la vez, un módulo importable **y** un script ejecutable. El archivo `script_test.py` de esta carpeta contiene:

```python
def func_saludar(name: str):
    print(f"Hola {name}")

func_saludar(name="Heber")

print(__name__)
```

Si lo ejecutas como script (`python script_test.py`), imprime `Hola Heber` y luego `__main__`. Pero si lo **importas**, ocurre lo mismo — el `print` se ejecuta igual, porque no hay ninguna condición que lo evite:

In [ ]:
import script_test  # se ejecuta la llamada a func_saludar() aunque solo lo estemos IMPORTANDO

print("__name__ del módulo importado:", script_test.__name__)

Esto casi nunca es lo que quieres: importar un módulo no debería tener efectos secundarios inesperados (como imprimir cosas o, peor, disparar un cálculo). La solución es el patrón `if __name__ == "__main__":`, presente en `script_test_2.py`:

```python
def func_saludar(name: str):
    print(f"Hola {name}")


if __name__ == "__main__":
    func_saludar(name="Heber")
```

Cuando Python **importa** un archivo, fija `__name__` con el nombre del módulo (`"script_test_2"`). Cuando lo **ejecuta directamente** como script, fija `__name__ = "__main__"`. Por eso `func_saludar` solo se llama al ejecutar el archivo directamente, nunca al importarlo:

In [ ]:
import script_test_2  # no imprime nada: __name__ vale "script_test_2", no "__main__"

print("__name__ del módulo importado:", script_test_2.__name__)

> 📊 En Excel esto sería como una macro que se dispara sola al **abrir** el archivo, incluso si solo querías consultarlo como referencia desde otro libro con `='[Otro.xlsx]Hoja1'!A1`. El patrón `if __name__ == "__main__":` es el equivalente a decir "esta macro solo se ejecuta si abres este archivo directamente, no si otro libro solo lee un valor de él".

## 6. Paquetes de Python

Un **paquete** agrupa varios módulos relacionados en una carpeta, usando notación de puntos para acceder a ellos. Igual que los módulos evitan colisiones entre nombres de variables, los paquetes evitan colisiones entre nombres de módulos.

En esta carpeta tienes `paquete_test/`, con esta estructura:

```
paquete_test/
├── api_modelo.py     -> class APIMod
├── base_datos.py     -> class Conexion, class BaseDatos
└── estimador.py      -> class Estimador
```

Como la carpeta `paquete_test` está en el mismo directorio que este notebook (y por tanto en `sys.path`), se puede acceder a sus módulos con notación de punto:

In [ ]:
import paquete_test.estimador

estimador = paquete_test.estimador.Estimador()
estimador.predict(y=[10.0, 20.0, 30.0])

Importar solo el paquete (`import paquete_test`), sin más, **no** coloca sus módulos en el *namespace* — es sintácticamente correcto pero prácticamente inútil:

In [ ]:
import paquete_test

try:
    paquete_test.api_modelo
except AttributeError as e:
    print(f"{type(e).__name__}: {e}")

import paquete_test.api_modelo  # ahora sí

paquete_test.api_modelo.APIMod(url="127.0.0.1", port=5000)

### Inicializar un paquete: `__init__.py` y `__all__`

Si añades un archivo `__init__.py` dentro de la carpeta del paquete, Python lo ejecuta automáticamente al importar el paquete (o cualquiera de sus módulos). Sirve para "personalizar" la inicialización — típicamente, para exponer las clases más importantes directamente bajo el nombre del paquete.

`paquete_test_2/` tiene la misma estructura que `paquete_test/`, más un `__init__.py`:

```python
from paquete_test_2.base_datos import Conexion, BaseDatos
from paquete_test_2.api_modelo import APIMod
from paquete_test_2.estimador import Estimador

__all__ = ["Conexion", "BaseDatos", "APIMod", "Estimador"]
```

`__all__` es la lista de nombres que se exportan cuando alguien hace `from paquete_test_2 import *`. Documenta explícitamente cuál es la interfaz pública del paquete.

In [ ]:
import paquete_test_2

# Gracias al __init__.py, Estimador está disponible directamente bajo el paquete,
# sin necesidad de "paquete_test_2.estimador.Estimador"
modelo = paquete_test_2.Estimador()
print(modelo.predict(y=[5.0]))

api = paquete_test_2.APIMod(url="127.0.0.1", port=5000)
print(api.url, api.port)

**Para ti:** compara `paquete_test` (sin `__init__.py`) con `paquete_test_2` (con `__init__.py`). Escribe en una celda de texto, con tus palabras, qué línea de código cambia para acceder a `Estimador` en cada caso y por qué.

_Tu respuesta aquí._

## 7. PEP 8: convenciones de estilo

PEP 8 es la guía de estilo oficial de Python. No es opcional en un entorno profesional: código que "funciona" pero no sigue PEP 8 es más difícil de leer, revisar y mantener en equipo — y la mayoría de herramientas (linters, formateadores, revisores de PR) lo dan por hecho.

> 📊 En Excel esto sería como acordar con tu equipo nombrar los rangos con nombres descriptivos (`Tabla_Primas_2025`) en vez de dejar referencias sueltas (`$B$2:$B$50`): cualquiera que abra el libro por primera vez entiende qué es cada cosa sin tener que rastrear celda por celda.

Puntos clave (no es una lista exhaustiva — la referencia completa está en [PEP 8](https://peps.python.org/pep-0008/)):

**Nombrado**
- Variables y funciones: `snake_case` (`prima_neta`, `calcular_recargo`).
- Clases: `CapWords` / `PascalCase` (`Poliza`, `GestorDeSiniestros`).
- Constantes: `MAYUSCULAS_CON_GUION_BAJO` (`TIPO_IVA`, `RECARGO_MAXIMO`).
- Nombres "privados" de uso interno: un guion bajo delante (`_calculo_interno`).

**Espaciado**
- 4 espacios de indentación, nunca tabuladores.
- Un espacio alrededor de operadores (`prima = base * (1 + recargo)`, no `prima=base*(1+recargo)`).
- Sin espacio justo dentro de paréntesis/corchetes (`func(x)`, no `func( x )`).
- Dos líneas en blanco entre funciones/clases de nivel superior, una entre métodos de una clase.

**Longitud de línea**
- Máximo recomendado: 79 caracteres (algunos equipos relajan a 99-100 para código, manteniendo 79 para docstrings/comentarios). Si una línea se alarga demasiado, es también una señal de que quizá conviene dividir la expresión.

**Imports**
- Uno por línea, agrupados en este orden: librería estándar, librerías de terceros, código propio — con una línea en blanco entre grupos.

In [ ]:
# Mal (no PEP 8): nombres en camelCase, sin espacios, sin tipado, línea de import mezclada
codigo_mal = '''
import pandas as pd,numpy as np
def calculaPrimaTotal(primaNeta,Recargo):
    return primaNeta*(1+Recargo)
'''

# Bien (PEP 8): snake_case, espaciado, tipado, imports separados
codigo_bien = '''
import numpy as np
import pandas as pd


def calcular_prima_total(prima_neta: float, recargo: float) -> float:
    """Calcula la prima total aplicando un recargo sobre la prima neta."""
    return prima_neta * (1 + recargo)
'''

print(codigo_mal)
print("---")
print(codigo_bien)

> ⚠️ **Error típico**: mezclar tabuladores y espacios para indentar. En un editor puede *verse* alineado y aun así Python lanzar `TabError: inconsistent use of tabs and spaces`. Configura tu editor para que la tecla Tab inserte siempre espacios (4, por convención) — Jupyter y VS Code lo hacen así por defecto.

En la práctica, en un proyecto real no se revisa PEP 8 a ojo: se usan herramientas automáticas como `black` (formateador) o `ruff`/`flake8` (linters) que señalan o corrigen estas cosas por ti. No hace falta memorizar la guía completa — sí interiorizar el criterio de fondo: **código legible para otra persona (o para ti mismo dentro de seis meses), no solo código que funciona**.

**Para ti:** el siguiente fragmento no sigue PEP 8. Reescríbelo en la celda de código siguiente aplicando nombrado, espaciado y tipado correctos (puedes añadir un docstring breve).

```python
def CalculaRecargoPorSiniestros(NumSin,primaBase):
  if NumSin>3:
    return primaBase*1.5
  else:
    return primaBase
```

In [ ]:
# Tu código aquí


## Resumen

- Un **módulo** es un archivo `.py`; un **paquete** es una carpeta de módulos relacionados (con o sin `__init__.py`).
- Python busca los módulos a importar recorriendo `sys.path`: el directorio actual, `PYTHONPATH`, y los directorios de la instalación.
- `import x`, `from x import y`, y sus variantes con `as` hacen cosas distintas con el *namespace* — `from x import *` es mala práctica casi siempre.
- `if __name__ == "__main__":` distingue "este archivo se está ejecutando directamente" de "este archivo se está importando" — evita efectos secundarios inesperados al importar.
- `__init__.py` personaliza la inicialización de un paquete y, junto con `__all__`, documenta su interfaz pública.
- PEP 8 fija convenciones de nombrado (`snake_case`/`CapWords`/`MAYUSCULAS`), espaciado y longitud de línea — no es estético, es lo que hace el código legible y revisable en equipo.
- En proyectos reales, herramientas como `black`/`ruff` automatizan la comprobación de PEP 8: no hace falta hacerlo a mano línea por línea.

**Siguiente:** `02_teoria_excepciones.ipynb`, donde veremos cómo gestionar errores de forma robusta con `try`/`except`/`else`/`finally`.